# Transformer Block Coupling and Cross-Token Influence Metrics
### Self-Contained Demo (No Package Import Required)

This notebook demonstrates the computation and visualization of:
1. **Block Coupling Metrics** - measuring alignment between layer Jacobians (ICLR 2025)
2. **Cross-Token Influence Metrics** - measuring how each input token influences the output

## Mathematical Background

### Block Coupling
For residual blocks $X^{l+1} = X^l + f^{l+1}(X^l)$, we compute:
- Jacobians: $J^l = \frac{\partial f^l(X^{l-1})}{\partial X^{l-1}}$
- SVD: $J_1 = U_1 S_1 V_1^T$, $J_2 = U_2 S_2 V_2^T$
- Coupling: $m_K(J_1, J_2) = \frac{||U_2^T J_1 V_2 - S_1||_F}{||s_1||_p}$

### Cross-Token Influence
For each layer $l$ and input token $j$, compute residual Jacobian:
$$J_{residual} = \frac{\partial f^l(X^{l-1})_T}{\partial x_j^{l-1}}$$

Then measure via:
- Frobenius norm: $||J||_F = \sqrt{\sum S^2}$
- Spectral norm: $\max(S)$
- Participation ratio: $(\sum S)^2 / \sum S^2$
- Entropy effective rank: $\exp(-\sum p_i \log p_i)$ where $p_i = s_i / \sum s_i$

## 1. Imports and Setup

In [ ]:
# Standard library
import os
import time
from datetime import datetime
from collections import defaultdict

# PyTorch
import torch
from torch.autograd import grad
from functorch.experimental import chunk_vmap

# HuggingFace
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Visualization
import matplotlib.pyplot as plt
import numpy as np

# Set plotting style
plt.style.use('default')
%matplotlib inline

# Check GPU availability
if torch.cuda.is_available():
    print("✓ GPU is available!")
    print(f"  Device: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU not available - will use CPU (slower)")

print("\n✓ All libraries imported successfully!")

## 2. Helper Functions

In [ ]:
def timestamp(*args):
    """
    Print timestamped message for logging.
    """
    now = datetime.now()
    current_time = now.strftime("%H:%M:%S")
    label = " ".join(map(str, args))
    print(str(current_time) + ": " + label, flush=True)

In [ ]:
def jacobian(output, input, index, chunks, index_in=None, verbose=False, device="cuda"):
    """
    Computes the Jacobian of `d{output}/d{input}` from transformer hooks
    by vectorizing over gradients.

    output:     Jacobian wrt this output
    input:      Jacobian wrt this input
    index:      index of output token
    chunks:     number of chunks used to vectorize Jacobian computation
    index_in:   (optional) changes input token of Jacobian if not `index`
    """
    tt = time.time()

    output = output[0, index, :]
    I_N = torch.eye(output.numel()).to(device)

    index_in = index_in if index_in is not None else index
    def get_vjp(v):
        return grad(output, input, v, retain_graph=True)[0][0, index_in, :]

    jacobian = chunk_vmap(get_vjp, chunks=chunks)(I_N)
    print(f"Jacobian computed in {time.time()-tt:.3f} seconds") if verbose else None

    return jacobian

In [ ]:
def randomevd(mm, d, k, l, its=0, device='cuda'):
    """
    Approximate top eigenvalues/vectors using randomized algorithm.
    
    mm:     matrix-matrix multiply function
    d:      dimension
    k:      number of eigenvalues to keep
    l:      oversampling parameter
    its:    number of power iterations
    """
    Q = torch.randn(d, l, device=device)
    
    # Power iteration
    for _ in range(its):
        Q = mm(Q)
        Q, _ = torch.linalg.qr(Q)
    
    # Rayleigh quotient
    R = Q.T @ mm(Q)
    R = (R + R.T) / 2  # Symmetrize
    
    # Eigendecomposition
    D, S = torch.linalg.eigh(R)
    V = Q @ S
    
    # Return top k (flip to descending order)
    return torch.flip(D, [0])[:k], torch.flip(V, [1])[:, :k]


def randomsvd_each(J, K, L, E, ITS):
    """
    Compute randomized SVD for a single Jacobian.
    """
    device = J.device
    m, n = J.shape
    k = K + E
    l = K + E + L
    
    # Left singular vectors
    def mml(Q):
        return J @ (J.T @ Q)
    Dl, U = randomevd(mml, m, k, l, its=ITS, device=device)
    
    # Right singular vectors
    def mmr(Q):
        return J.T @ (J @ Q)
    Dr, V = randomevd(mmr, n, k, l, its=ITS, device=device)
    
    # Singular values
    S = Dl[:K] ** 0.5
    
    return U[:, :K], S, V[:, :K].T


def randomsvd(J, K, L, E, ITS):
    """
    Compute randomized SVD for a list of Jacobians.
    """
    Us, Ss, Vs = [], [], []
    for j in range(len(J)):
        U, S, V = randomsvd_each(J[j], K, L, E, ITS)
        Us.append(U)
        Ss.append(S)
        Vs.append(V)
    return Us, Ss, Vs


def svd(Jac, method="torch", K=50, L=20, E=5, ITS=20, verbose=False):
    """
    Computes the SVD of a list of Jacobians.

    Jac:        list of Jacobians
    method:
    -   "torch" calls `torch.linalg.svd()`
    -   "random" calls random SVD (implemented above),
        - if using "random", `K, L, E, ITS` will be used
    """
    if method == "torch":
        Us, Ss, Vs = [], [], []
        for j in range(len(Jac)):
            i = j
            tt = time.time()
            timestamp('svding layer {}'.format(i))
            U, S, V = torch.linalg.svd(Jac[i])
            Us.append(U)
            Ss.append(S)
            Vs.append(V)
            print(f"SVD computed in {time.time()-tt:.3f} seconds") if verbose else None
    elif method == "random":
        Us, Ss, Vs = randomsvd(Jac, K=K, L=L, E=E, ITS=ITS)
    else:
        raise NotImplementedError("Only torch and random SVD are implemented")
    return Us, Ss, Vs

In [ ]:
def diag_sv_trace_similarity(J1, S1, U2, V2, p=2):
    """
    Compute coupling metric between J1 and singular vectors of J2.
    
    Returns both trace-normalized and p-norm-normalized versions.
    Swap U2 and V2 for the vju variant.
    """
    M = U2.T @ J1 @ V2
    tr = torch.trace(S1)
    norm = torch.norm(torch.diag(S1), p=p)
    diff = torch.linalg.norm(torch.abs(M) - S1)
    return diff / tr, diff / norm

In [ ]:
def metrics(Jac, Us=None, Ss=None, Vs=None, p=2, num_sing_vecs=(10,30,50),
            svd_method="torch", L=20, E=5, ITS=20, device="cpu", verbose=False):
    """
    Compute coupling metrics between all pairs of layers.
    """
    aln_ujv_all_k = {}
    aln_vju_all_k = {}

    # Do torch SVD by default if not provided
    if Us is None or Ss is None or Vs is None:
        Us, Ss, Vs = svd(Jac, method=svd_method, K=max(num_sing_vecs), L=L, E=E, ITS=ITS, verbose=verbose)

    S = torch.stack(Ss).cpu()
    U_all = [u.to(device) for u in Us]
    V_all = [v.to(device) for v in Vs]
    J = [j.to(device) for j in Jac]

    for K in num_sing_vecs:
        U, V = [u[:, :K] for u in U_all], [v[:K, :].T for v in V_all]

        ujv_mat_trace = torch.zeros((len(S), len(S)))
        vju_mat_trace = torch.zeros((len(S), len(S)))

        ujv_mat_norm = torch.zeros((len(S), len(S)))
        vju_mat_norm = torch.zeros((len(S), len(S)))

        for i in range(len(S)):
            for j in range(len(S)):
                uj, ji, vj = U[j], J[i], V[j]
                if uj.shape[0] != ji.shape[1] or vj.shape[0] != ji.shape[0]:
                    print("wrong shape")
                    continue

                S_i = torch.diag(S[i][:K])

                ujv_mat_trace[i, j], ujv_mat_norm[i, j] = diag_sv_trace_similarity(ji, S_i, uj, vj, p=p)
                vju_mat_trace[i, j], vju_mat_norm[i, j] = diag_sv_trace_similarity(ji, S_i, vj, uj, p=p)

        aln_ujv_all = {}
        aln_ujv_all['trace'] = ujv_mat_trace
        aln_ujv_all['norm'] = ujv_mat_norm

        aln_vju_all = {}
        aln_vju_all['trace'] = vju_mat_trace
        aln_vju_all['norm'] = vju_mat_norm

        aln_ujv_all_k[K] = aln_ujv_all
        aln_vju_all_k[K] = aln_vju_all

    return aln_ujv_all_k, aln_vju_all_k

## 3. Block Coupling Functions

In [ ]:
def coupling_from_hooks(hooks, p=2, num_sing_vecs=(10,30,50), index=-1, index_in=None,
                        activation=None, chunks=4, verbose=False, device="cuda"):
    """
    Computes the coupling of residual Jacobians across hooks.

    hooks:      dict of representations before and after skip connection
                - hooks[layer] = {0: x_in, 1: x_out}
    p:          order of p-norm for coupling measurement
    num_sing_vecs: number of top singular vectors to use in computing coupling
    index:      output token index for Jacobian
    index_in:   input token index for Jacobian (defaults to `index`)
    activation: specifies whether to apply activation to `x_out` before computing Jacobian
    chunks:     number of chunks in Jacobian computation
    """
    Jac = []

    for h in hooks:
        timestamp("computing J of: ", h) if verbose else None

        x_in = hooks[h][0]
        x_out = hooks[h][1]
        dim = x_out.shape[-1]

        if activation is None:
            J = jacobian(x_out, x_in, index=-1, device=device, chunks=chunks).detach()
            Jac.append(J - torch.eye(dim).to(device))
            timestamp("Jacobian shape ", J.shape) if verbose else None
        else:
            J = jacobian(activation(x_out), x_in, index=index, index_in=index_in, device=device, chunks=chunks).detach()
            Jac.append(J - torch.eye(dim).to(device))

    timestamp("Computing coupling metrics") if verbose else None

    Us, Ss, Vs = svd(Jac)
    coupling_ujv, coupling_vju = metrics(Jac, Us, Ss, Vs, p=p, num_sing_vecs=num_sing_vecs)

    return coupling_ujv, coupling_vju

In [ ]:
def run_coupling_hf(model, tokenizer, model_name, prompts, start=None, end=None,
                    save=False, out_path=None, device="cuda", verbose=False):
    """
    Runs coupling experiment for HuggingFace model and tokenizer.

    model:      model following HuggingFace API
    tokenizer:  tokenizer of model
    model_name: model name, e.g. `Meta-Llama-3-8B`
    prompts:    list of str prompts to pass to tokenizer
    """
    out = defaultdict(dict)
    start = start if start is not None else 0
    end = end if end is not None else len(prompts)

    for i, prompt in zip(range(start, end), prompts):
        timestamp(f"Running prompt {i + 1} of {end}")

        out[i] = {"prompt": prompt}
        print(prompt) if verbose else None

        tokens = tokenizer(prompt, return_tensors='pt')
        input_ids = tokens.input_ids
        num_tokens = input_ids.shape[1]
        print("Number of tokens:", num_tokens) if verbose else None

        chunks = 2 * (num_tokens // 20) + 5 + i
        print("Number of chunks:", chunks) if verbose else None

        input_ids = input_ids.to(device)
        outputs = model(input_ids, output_hidden_states=True)
        L = len(outputs.hidden_states) - 1

        # Format as hooks
        outputs_zip = {}
        for j in range(L):
            outputs_zip[f"block_{j}"] = {0: outputs.hidden_states[j], 1: outputs.hidden_states[j+1]}

        # Compute coupling
        coupling_ujv, coupling_vju = coupling_from_hooks(
            outputs_zip, activation=None, chunks=chunks, verbose=verbose, device=device
        )

        out[i]["coupling_ujv"] = coupling_ujv
        out[i]["coupling_vju"] = coupling_vju
        timestamp(f"Ended prompt") if verbose else None

    if save:
        if out_path is None:
            out_path = os.getcwd()
            print(f"Saving enabled but out_path not specified. Saving in `{out_path}`")
        out_file = os.path.join(out_path, "_".join((model_name, "coupling.pt")))
        torch.save(out, out_file)

    return out

## 4. Cross-Token Influence Functions

In [ ]:
def cross_token_influence_from_hooks(hooks, index=-1, chunks=4, verbose=False, device="cuda"):
    """
    Compute cross-token influence metrics for each layer.
    
    For each layer and input token j, computes influence from token j to output token T.
    Returns four metrics: Frobenius norm, spectral norm, participation ratio, entropy effective rank.
    """
    # Get sequence length from first hook
    first_hook = list(hooks.keys())[0]
    seq_len = hooks[first_hook][0].shape[1]
    
    # Convert negative index to positive
    T = seq_len + index if index < 0 else index
    
    num_layers = len(hooks)
    
    # Initialize output tensors on GPU [num_layers, T+1]
    frobenius_norm = torch.zeros(num_layers, T + 1, device=device)
    spectral_norm = torch.zeros(num_layers, T + 1, device=device)
    participation_ratio = torch.zeros(num_layers, T + 1, device=device)
    entropy_effective_rank = torch.zeros(num_layers, T + 1, device=device)
    
    for layer_idx, h in enumerate(hooks):
        timestamp(f"Computing cross-token influence for layer {layer_idx}") if verbose else None
        
        x_in = hooks[h][0]
        x_out = hooks[h][1]
        dim = x_out.shape[-1]
        
        # Compute influence from each input token j to output token T
        for j in range(T + 1):
            # Compute Jacobian for this token pair
            J = jacobian(x_out, x_in, index=index, index_in=j, device=device, chunks=chunks, verbose=False).detach()
            J_residual = J - torch.eye(dim).to(device)
            
            # Compute SVD
            _, S, _ = torch.linalg.svd(J_residual)
            
            # Compute metrics from singular values
            # 1. Frobenius norm: sqrt(sum(S^2))
            frobenius_norm[layer_idx, j] = torch.sqrt(torch.sum(S ** 2))
            
            # 2. Spectral norm: max(S)
            spectral_norm[layer_idx, j] = S[0]
            
            # 3. Participation ratio: (sum(S))^2 / sum(S^2)
            sum_S = torch.sum(S)
            sum_S2 = torch.sum(S ** 2)
            participation_ratio[layer_idx, j] = (sum_S ** 2) / sum_S2 if sum_S2 > 0 else 0
            
            # 4. Entropy effective rank: exp(-sum(p * log(p)))
            p = S / sum_S if sum_S > 0 else torch.zeros_like(S)
            p = p[p > 0]  # Filter out zeros for log
            entropy = -torch.sum(p * torch.log(p))
            entropy_effective_rank[layer_idx, j] = torch.exp(entropy)
    
    # Move results to CPU before returning
    return {
        'frobenius_norm': frobenius_norm.cpu(),
        'spectral_norm': spectral_norm.cpu(),
        'participation_ratio': participation_ratio.cpu(),
        'entropy_effective_rank': entropy_effective_rank.cpu()
    }

In [ ]:
def run_cross_token_influence_hf(model, tokenizer, model_name, prompts, start=None, end=None,
                                 save=False, out_path=None, device="cuda", verbose=False):
    """
    Runs cross-token influence experiment for HuggingFace model and tokenizer.
    """
    out = defaultdict(dict)
    start = start if start is not None else 0
    end = end if end is not None else len(prompts)

    for i, prompt in zip(range(start, end), prompts):
        timestamp(f"Running prompt {i + 1} of {end}")

        out[i] = {"prompt": prompt}
        print(prompt) if verbose else None

        tokens = tokenizer(prompt, return_tensors='pt')
        input_ids = tokens.input_ids
        num_tokens = input_ids.shape[1]
        print("Number of tokens:", num_tokens) if verbose else None

        chunks = 2 * (num_tokens // 20) + 5 + i
        print("Number of chunks:", chunks) if verbose else None

        input_ids = input_ids.to(device)
        outputs = model(input_ids, output_hidden_states=True)
        L = len(outputs.hidden_states) - 1

        # Format as hooks
        outputs_zip = {}
        for j in range(L):
            outputs_zip[f"block_{j}"] = {0: outputs.hidden_states[j], 1: outputs.hidden_states[j+1]}

        # Compute cross-token influence
        influence_metrics = cross_token_influence_from_hooks(
            outputs_zip, chunks=chunks, verbose=verbose, device=device
        )

        out[i].update(influence_metrics)
        timestamp(f"Ended prompt") if verbose else None

    if save:
        if out_path is None:
            out_path = os.getcwd()
            print(f"Saving enabled but out_path not specified. Saving in `{out_path}`")
        out_file = os.path.join(out_path, "_".join((model_name, "cross_token_influence.pt")))
        torch.save(out, out_file)

    return out

## 5. Visualization Functions

In [ ]:
def plot_coupling_matrix(coupling_dict, K=10, prompt_idx=0, variant='ujv'):
    """
    Plot the coupling matrix as a heatmap.
    
    Args:
        coupling_dict: Results from run_coupling_hf
        K: Number of singular vectors (10, 30, or 50)
        prompt_idx: Which prompt to visualize
        variant: 'ujv' or 'vju'
    """
    key = f"coupling_{variant}"
    
    # Extract coupling matrix (using norm-normalized version)
    coupling_matrix = coupling_dict[prompt_idx][key][K]['norm'].cpu().numpy()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot full matrix
    im1 = ax1.imshow(coupling_matrix, cmap='viridis', aspect='auto')
    ax1.set_xlabel('Layer j', fontsize=12)
    ax1.set_ylabel('Layer i', fontsize=12)
    ax1.set_title(f'Block Coupling Matrix ({variant.upper()}, K={K})\n' + 
                  f'Prompt: "{coupling_dict[prompt_idx]["prompt"][:50]}..."', 
                  fontsize=14)
    plt.colorbar(im1, ax=ax1, label='Coupling (lower = more aligned)')
    
    # Plot diagonal and off-diagonal separately
    num_layers = coupling_matrix.shape[0]
    layers = np.arange(num_layers)
    
    # Diagonal (self-coupling)
    diagonal = np.diag(coupling_matrix)
    ax2.plot(layers, diagonal, 'o-', label='Self-coupling (diagonal)', linewidth=2)
    
    # Mean off-diagonal per layer
    off_diag_mean = []
    for i in range(num_layers):
        mask = np.ones(num_layers, dtype=bool)
        mask[i] = False
        off_diag_mean.append(coupling_matrix[i, mask].mean())
    
    ax2.plot(layers, off_diag_mean, 's-', label='Mean cross-layer coupling', linewidth=2, alpha=0.7)
    ax2.set_xlabel('Layer', fontsize=12)
    ax2.set_ylabel('Coupling', fontsize=12)
    ax2.set_title('Coupling vs Layer Depth', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nCoupling Statistics (K={K}, {variant.upper()}):")
    print(f"  Min coupling: {coupling_matrix.min():.4f}")
    print(f"  Max coupling: {coupling_matrix.max():.4f}")
    print(f"  Mean coupling: {coupling_matrix.mean():.4f}")
    print(f"  Mean diagonal: {diagonal.mean():.4f}")
    print(f"  Mean off-diagonal: {np.mean(off_diag_mean):.4f}")

In [ ]:
def compare_K_values(coupling_dict, prompt_idx=0, variant='ujv'):
    """
    Compare coupling matrices for different K values.
    """
    K_values = [10, 30, 50]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, K in enumerate(K_values):
        coupling_matrix = coupling_dict[prompt_idx][f"coupling_{variant}"][K]['norm'].cpu().numpy()
        
        im = axes[idx].imshow(coupling_matrix, cmap='viridis', aspect='auto')
        axes[idx].set_xlabel('Layer j', fontsize=11)
        axes[idx].set_ylabel('Layer i', fontsize=11)
        axes[idx].set_title(f'K={K}\nMean: {coupling_matrix.mean():.4f}', fontsize=12)
        plt.colorbar(im, ax=axes[idx])
    
    fig.suptitle(f'Coupling Matrices for Different K Values ({variant.upper()})', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_cross_token_influence(influence_dict, tokenizer, prompt_idx=0):
    """
    Plot all four cross-token influence metrics.
    """
    prompt = influence_dict[prompt_idx]['prompt']
    
    # Get token strings
    tokens = tokenizer(prompt, return_tensors='pt')
    token_ids = tokens.input_ids[0]
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    
    metrics = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, (metric_key, metric_name) in enumerate(metrics.items()):
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        
        # Plot heatmap
        im = axes[idx].imshow(data, cmap='viridis', aspect='auto')
        axes[idx].set_xlabel('Input Token Position', fontsize=11)
        axes[idx].set_ylabel('Layer', fontsize=11)
        axes[idx].set_title(f'{metric_name}\nHigher = stronger influence', fontsize=12)
        
        # Add token labels on x-axis
        axes[idx].set_xticks(range(len(token_strs)))
        axes[idx].set_xticklabels(token_strs, rotation=45, ha='right', fontsize=9)
        
        plt.colorbar(im, ax=axes[idx])
    
    fig.suptitle(f'Cross-Token Influence: "{prompt[:60]}..."', fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_influence_by_layer(influence_dict, tokenizer, prompt_idx=0):
    """
    Plot how influence from different tokens changes across layers.
    """
    prompt = influence_dict[prompt_idx]['prompt']
    tokens = tokenizer(prompt, return_tensors='pt')
    token_strs = [tokenizer.decode([tid]) for tid in tokens.input_ids[0]]
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()
    
    metrics = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    for idx, (metric_key, metric_name) in enumerate(metrics.items()):
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        num_layers, num_tokens = data.shape
        
        # Plot each token's influence across layers
        for token_idx in range(num_tokens):
            axes[idx].plot(
                range(num_layers),
                data[:, token_idx],
                label=f'Token {token_idx}: {token_strs[token_idx][:10]}',
                marker='o' if num_tokens <= 5 else None,
                markersize=3,
                linewidth=2,
                alpha=0.7
            )
        
        axes[idx].set_xlabel('Layer', fontsize=11)
        axes[idx].set_ylabel(metric_name, fontsize=11)
        axes[idx].set_title(f'{metric_name} vs Layer Depth', fontsize=12)
        axes[idx].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        axes[idx].grid(True, alpha=0.3)
    
    fig.suptitle(f'Token Influence Across Layers: "{prompt[:60]}..."', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
def analyze_token_importance(influence_dict, tokenizer, prompt_idx=0):
    """
    Analyze which tokens have the most influence on average.
    """
    prompt = influence_dict[prompt_idx]['prompt']
    tokens = tokenizer(prompt, return_tensors='pt')
    token_strs = [tokenizer.decode([tid]) for tid in tokens.input_ids[0]]
    
    # Average across all layers for each token
    metrics = ['frobenius_norm', 'spectral_norm', 'participation_ratio', 'entropy_effective_rank']
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    metric_names = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    for idx, metric_key in enumerate(metrics):
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        
        # Mean influence across layers for each token
        mean_influence = data.mean(axis=0)
        std_influence = data.std(axis=0)
        
        x_pos = np.arange(len(token_strs))
        axes[idx].bar(x_pos, mean_influence, yerr=std_influence, 
                     capsize=5, alpha=0.7, color='steelblue')
        axes[idx].set_xlabel('Token', fontsize=11)
        axes[idx].set_ylabel(f'Mean {metric_names[metric_key]}', fontsize=11)
        axes[idx].set_title(f'Average {metric_names[metric_key]} per Token\n(across all layers)', 
                           fontsize=12)
        axes[idx].set_xticks(x_pos)
        axes[idx].set_xticklabels(token_strs, rotation=45, ha='right', fontsize=9)
        axes[idx].grid(True, alpha=0.3, axis='y')
    
    fig.suptitle(f'Token Importance Analysis: "{prompt[:60]}..."', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print most influential tokens
    print("\nMost Influential Tokens (by Frobenius norm):")
    frob_data = influence_dict[prompt_idx]['frobenius_norm'].cpu().numpy()
    mean_frob = frob_data.mean(axis=0)
    sorted_indices = np.argsort(mean_frob)[::-1]
    for i, idx in enumerate(sorted_indices[:min(5, len(token_strs))]):
        print(f"  {i+1}. Token {idx} ('{token_strs[idx]}'): {mean_frob[idx]:.4f}")

In [ ]:
def analyze_layer_statistics(influence_dict, prompt_idx=0):
    """
    Analyze how influence statistics change across layers.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    metrics = ['frobenius_norm', 'spectral_norm', 'participation_ratio', 'entropy_effective_rank']
    metric_names = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    for idx, metric_key in enumerate(metrics):
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        num_layers = data.shape[0]
        
        # Statistics per layer
        layer_mean = data.mean(axis=1)
        layer_std = data.std(axis=1)
        layer_max = data.max(axis=1)
        layer_min = data.min(axis=1)
        
        layers = np.arange(num_layers)
        
        # Plot mean with std band
        axes[idx].plot(layers, layer_mean, 'o-', label='Mean', linewidth=2, color='steelblue')
        axes[idx].fill_between(layers, layer_mean - layer_std, layer_mean + layer_std, 
                              alpha=0.3, color='steelblue', label='±1 std')
        axes[idx].plot(layers, layer_max, 's--', label='Max', linewidth=1.5, 
                      alpha=0.6, color='darkgreen')
        axes[idx].plot(layers, layer_min, '^--', label='Min', linewidth=1.5, 
                      alpha=0.6, color='darkred')
        
        axes[idx].set_xlabel('Layer', fontsize=11)
        axes[idx].set_ylabel(metric_names[metric_key], fontsize=11)
        axes[idx].set_title(f'{metric_names[metric_key]} Statistics by Layer', fontsize=12)
        axes[idx].legend(loc='best', fontsize=9)
        axes[idx].grid(True, alpha=0.3)
    
    prompt = influence_dict[prompt_idx]['prompt']
    fig.suptitle(f'Layer-wise Statistics: "{prompt[:60]}..."', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
def compare_prompts_influence(influence_dict, tokenizer, metric_key='frobenius_norm'):
    """
    Compare a specific metric across different prompts.
    """
    num_prompts = len(influence_dict)
    fig, axes = plt.subplots(1, num_prompts, figsize=(6*num_prompts, 5))
    
    if num_prompts == 1:
        axes = [axes]
    
    metric_names = {
        'frobenius_norm': 'Frobenius Norm',
        'spectral_norm': 'Spectral Norm',
        'participation_ratio': 'Participation Ratio',
        'entropy_effective_rank': 'Entropy Effective Rank'
    }
    
    for prompt_idx in range(num_prompts):
        prompt = influence_dict[prompt_idx]['prompt']
        tokens = tokenizer(prompt, return_tensors='pt')
        token_strs = [tokenizer.decode([tid]) for tid in tokens.input_ids[0]]
        
        data = influence_dict[prompt_idx][metric_key].cpu().numpy()
        
        im = axes[prompt_idx].imshow(data, cmap='viridis', aspect='auto')
        axes[prompt_idx].set_xlabel('Input Token', fontsize=10)
        axes[prompt_idx].set_ylabel('Layer', fontsize=10)
        axes[prompt_idx].set_title(f'Prompt {prompt_idx}\n"{prompt[:40]}..."', fontsize=11)
        axes[prompt_idx].set_xticks(range(len(token_strs)))
        axes[prompt_idx].set_xticklabels(token_strs, rotation=45, ha='right', fontsize=8)
        plt.colorbar(im, ax=axes[prompt_idx])
    
    fig.suptitle(f'{metric_names[metric_key]} Comparison', fontsize=14, y=1.00)
    plt.tight_layout()
    plt.show()

## 6. Load Model and Tokenizer

In [ ]:
# Model configuration
model_path = "gpt2"  # Start with gpt2 for testing
# model_path = "meta-llama/Meta-Llama-3-8B"  # Requires HF auth

model_name = os.path.normpath(os.path.basename(model_path))

# Load model with 4-bit quantization to save memory (if GPU available)
device = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(load_in_4bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="cuda",
        trust_remote_code=True,
        quantization_config=bnb_config
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True
    )

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
)

print(f"✓ Model loaded successfully!")
print(f"  Model name: {model_name}")
print(f"  Number of layers: {model.config.num_hidden_layers}")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Device: {device}")

if torch.cuda.is_available():
    print(f"\nGPU Memory Usage:")
    print(f"  Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"  Reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

## 7. Define Test Prompts

In [ ]:
prompts = [
    "What is the capital of France? The capital is",
    "The quick brown fox jumps over the lazy",
    "To be or not to be, that is the"
]

# Display tokenized prompts
print("Prompts and their tokens:\n")
for i, prompt in enumerate(prompts):
    tokens = tokenizer(prompt, return_tensors='pt')
    token_ids = tokens.input_ids[0]
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    print(f"Prompt {i}: {prompt}")
    print(f"  Tokens ({len(token_ids)}): {token_strs}")
    print()

## 8. Compute Block Coupling Metrics

⏱️ **Note**: This may take 5-15 minutes depending on model size and number of prompts.

In [ ]:
print("Computing block coupling metrics...")
print("This may take several minutes...\n")

coupling_results = run_coupling_hf(
    model, 
    tokenizer, 
    model_name, 
    prompts, 
    save=False, 
    verbose=True,
    device=device
)

print("\n✓ Block coupling computation complete!")

## 9. Visualize Block Coupling Results

In [ ]:
# Plot coupling matrix for first prompt
plot_coupling_matrix(coupling_results, K=10, prompt_idx=0, variant='ujv')

In [ ]:
# Compare different K values
compare_K_values(coupling_results, prompt_idx=0, variant='ujv')

## 10. Compute Cross-Token Influence Metrics

⏱️ **Note**: This may take 10-30 minutes depending on model size and sequence length.

In [ ]:
print("Computing cross-token influence metrics...")
print("This may take several minutes...\n")

influence_results = run_cross_token_influence_hf(
    model,
    tokenizer,
    model_name,
    prompts,
    save=False,
    verbose=True,
    device=device
)

print("\n✓ Cross-token influence computation complete!")

## 11. Visualize Cross-Token Influence Results

In [ ]:
# Plot all four metrics as heatmaps
plot_cross_token_influence(influence_results, tokenizer, prompt_idx=0)

In [ ]:
# Plot influence by layer
plot_influence_by_layer(influence_results, tokenizer, prompt_idx=0)

In [ ]:
# Analyze token importance
analyze_token_importance(influence_results, tokenizer, prompt_idx=0)

In [ ]:
# Analyze layer statistics
analyze_layer_statistics(influence_results, prompt_idx=0)

In [ ]:
# Compare metrics across prompts
compare_prompts_influence(influence_results, tokenizer, metric_key='frobenius_norm')

## 12. Summary and Interpretation

### Block Coupling Metrics
- **Lower values** = stronger alignment between layer Jacobians
- **Diagonal** shows self-coupling (should be near zero)
- **Off-diagonal** patterns reveal how layers share similar computational structure
- **K parameter** controls how many singular vectors are considered

### Cross-Token Influence Metrics
1. **Frobenius Norm**: Overall magnitude of influence (√Σ S²)
   - Higher = token has stronger overall influence on output
   - Captures total effect across all directions

2. **Spectral Norm**: Maximum directional influence (max singular value)
   - Higher = token has strong influence in at least one direction
   - Most sensitive to dominant effects

3. **Participation Ratio**: How evenly influence is distributed across dimensions
   - Higher = influence spread across many dimensions
   - Lower = influence concentrated in few dimensions

4. **Entropy Effective Rank**: Another measure of influence distribution (entropy-based)
   - Similar to participation ratio but uses entropy
   - Higher = more uniform distribution of influence

### Expected Patterns
- **Recent tokens** often show stronger influence (positional bias)
- **Content words** may show higher influence than function words
- **Deeper layers** may show different influence patterns than early layers
- **Participation ratio** indicates if influence is concentrated or spread out

## 13. Save Results (Optional)

In [ ]:
# Optionally save results
save_results = False  # Set to True to save

if save_results:
    output_dir = "./results"
    os.makedirs(output_dir, exist_ok=True)
    
    # Save coupling results
    coupling_file = os.path.join(output_dir, f"{model_name}_coupling_results.pt")
    torch.save(coupling_results, coupling_file)
    print(f"✓ Saved coupling results to {coupling_file}")
    
    # Save influence results
    influence_file = os.path.join(output_dir, f"{model_name}_influence_results.pt")
    torch.save(influence_results, influence_file)
    print(f"✓ Saved influence results to {influence_file}")
else:
    print("Results not saved (set save_results=True to save)")